# <span style="font-family: Arial, sans-serif; color:#97f788">xbooster</span>

## <span style="font-family: Arial, sans-serif; color:navyblue">Fine-Tuning - Getting started</span>

<span style="font-family: Arial, sans-serif; color:navyblue">Repo: <a href="https://github.com/xRiskLab/xBooster" title="GitHub link">https://github.com/xRiskLab/xBooster</a></span>

This notebook demonstrates how to fine-tune gradient boosted models and build scorecards
that distinguish between base and fine-tuned trees.

**Workflow:**

1. Train base models (XGBoost, LightGBM, CatBoost)
2. Fine-tune with `finetune_*` helpers
3. Build scorecards with `from_finetune_result()` or `n_base_trees`
4. Inspect `TreeSource` column and `summarize_score_sources()`
5. Score and compare distributions


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Generate synthetic credit data with ~25% default rate
np.random.seed(42)
n = 500
data = pd.DataFrame(
    {
        "income": np.random.lognormal(10.5, 0.5, n),
        "debt_ratio": np.random.beta(2, 5, n),
        "credit_age": np.random.exponential(5, n),
    }
)
# Create a signal strong enough for the models to learn meaningful splits
logit = -1 + 3 * data["debt_ratio"] - 0.0001 * data["income"] + 0.1 * data["credit_age"]
data["default"] = (np.random.rand(n) < 1 / (1 + np.exp(-logit))).astype(int)

X = data[["income", "debt_ratio", "credit_age"]]
y = data["default"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}, Event rate: {y.mean():.2%}")

Train: 350, Test: 150, Event rate: 6.40%


## 1. Train Base Models


In [2]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# XGBoost base model
xgb_base = XGBClassifier(
    n_estimators=10, max_depth=1, learning_rate=0.3, random_state=42, eval_metric="logloss"
)
xgb_base.fit(X_train, y_train)

# LightGBM base model
lgb_base = LGBMClassifier(
    n_estimators=10, max_depth=1, learning_rate=0.3, random_state=42, verbose=-1
)
lgb_base.fit(X_train, y_train)

# CatBoost base model
cb_base = CatBoostClassifier(iterations=10, depth=1, learning_rate=0.3, random_seed=42, verbose=0)
cb_base.fit(X_train, y_train)

print("Base models trained.")

Base models trained.


## 2. Fine-Tune with Same Features

When fine-tuning with the same feature set, base trees are frozen and new trees are appended.


In [3]:
from xbooster.finetuner import finetune_xgb, finetune_lgb, finetune_cb
from xbooster.xgb_constructor import XGBScorecardConstructor
from xbooster.lgb_constructor import LGBScorecardConstructor
from xbooster.cb_constructor import CBScorecardConstructor

# Same features: base trees are frozen, new trees appended
xgb_ft = finetune_xgb(xgb_base, X_train, y_train, n_estimators=5, learning_rate=0.1)
lgb_ft = finetune_lgb(lgb_base, X_train, y_train, n_estimators=5, learning_rate=0.1, verbose=-1)
cb_ft = finetune_cb(cb_base, X_train, y_train, n_estimators=5, learning_rate=0.1, verbose=0)

for name, ft in [("XGBoost", xgb_ft), ("LightGBM", lgb_ft), ("CatBoost", cb_ft)]:
    print(
        f"{name}: {ft.n_base_trees} base + {ft.n_total_trees - ft.n_base_trees} new = {ft.n_total_trees} total trees"
    )
    print(f"Base features: {ft.base_features}")
    print(f"New features: {ft.new_features}")

XGBoost: 10 base + 5 new = 15 total trees
Base features: ['income', 'debt_ratio', 'credit_age']
New features: []
LightGBM: 10 base + 5 new = 15 total trees
Base features: ['income', 'debt_ratio', 'credit_age']
New features: []
CatBoost: 10 base + 5 new = 15 total trees
Base features: ['income', 'debt_ratio', 'credit_age']
New features: []


## 3. Fine-Tune with Expanded Features (New Data Sources)

In production, new data sources often become available quarterly. For example, a bureau
adds `num_inquiries` or a new internal feature `savings_ratio` is engineered.

When new features are added, native continued training isn't possible (the base model
doesn't know about the new columns). Instead, `finetune_*` uses a **warm-start** approach:
the base model's raw predictions become the starting point for a fresh model trained on
all features. This means `n_base_trees=0` (no base trees carried over), but the base
model's knowledge is preserved via initial scores.


In [4]:
# Simulate new data arriving with 2 additional features that have predictive signal
np.random.seed(99)
X_train_expanded = X_train.copy()
X_train_expanded["num_inquiries"] = np.random.exponential(3, len(X_train))
X_train_expanded["savings_ratio"] = np.random.beta(5, 2, len(X_train))

X_test_expanded = X_test.copy()
X_test_expanded["num_inquiries"] = np.random.exponential(3, len(X_test))
X_test_expanded["savings_ratio"] = np.random.beta(5, 2, len(X_test))

# Augment the target: more inquiries and lower savings increase default risk
np.random.seed(99)
logit_new = (
    -2
    + 3 * X_train_expanded["debt_ratio"]
    - 0.0001 * X_train_expanded["income"]
    + 0.8 * X_train_expanded["num_inquiries"]
    - 2 * X_train_expanded["savings_ratio"]
)
y_train_new = pd.Series(
    (np.random.rand(len(X_train_expanded)) < 1 / (1 + np.exp(-logit_new))).astype(int),
    index=y_train.index,
)
print(f"Updated event rate: {y_train_new.mean():.2%}")

# Fine-tune XGBoost with expanded features (more trees to let all features surface)
xgb_ft_expanded = finetune_xgb(
    xgb_base, X_train_expanded, y_train_new, n_estimators=50, learning_rate=0.1
)

print(f"\nBase features: {xgb_ft_expanded.base_features}")
print(f"New features: {xgb_ft_expanded.new_features}")
print(f"All features: {xgb_ft_expanded.all_features}")
print(f"Base trees: {xgb_ft_expanded.n_base_trees} (warm-start: no base trees carried over)")
print(f"Total trees: {xgb_ft_expanded.n_total_trees}")

Updated event rate: 2.86%

Base features: ['income', 'debt_ratio', 'credit_age']
New features: ['num_inquiries', 'savings_ratio']
All features: ['income', 'debt_ratio', 'credit_age', 'num_inquiries', 'savings_ratio']
Base trees: 0 (warm-start: no base trees carried over)
Total trees: 50


In [5]:
# Build scorecard from expanded-feature model
xgb_sc_expanded = XGBScorecardConstructor(xgb_ft_expanded.model, X_train_expanded, y_train_new)
expanded_scorecard = xgb_sc_expanded.construct_scorecard()

# The scorecard now includes the new features
print("Features in expanded scorecard:", sorted(expanded_scorecard["Feature"].unique()))
expanded_scorecard[["Tree", "Feature", "Sign", "Split", "XAddEvidence", "IV"]]

Features in expanded scorecard: ['credit_age', 'income', 'num_inquiries', 'savings_ratio']


,Tree,Feature,Sign,Split,XAddEvidence,IV
0,0,num_inquiries,<,9.180827,-0.107105,1.959988
1,0,num_inquiries,>=,9.180827,0.316632,2.800504
2,1,num_inquiries,<,9.180827,-0.104820,1.959988
3,1,num_inquiries,>=,9.180827,0.262875,2.800504
4,2,num_inquiries,<,9.180827,-0.102628,1.959988
...,...,...,...,...,...,...
95,47,savings_ratio,>=,0.776701,-0.059653,0.752626
96,48,credit_age,<,10.023369,-0.023756,0.035995
97,48,credit_age,>=,10.023369,0.054828,0.137187
98,49,num_inquiries,<,9.180827,-0.041987,1.959988


## 4. Build Scorecards with `from_finetune_result()`


In [6]:
# Build scorecards from same-feature fine-tune results
xgb_sc = XGBScorecardConstructor.from_finetune_result(xgb_ft, X_train, y_train)
lgb_sc = LGBScorecardConstructor.from_finetune_result(lgb_ft, X_train, y_train)
cb_sc = CBScorecardConstructor.from_finetune_result(cb_ft, X_train, y_train)

# Construct and display XGBoost scorecard
xgb_scorecard = xgb_sc.construct_scorecard()
print("XGBoost scorecard columns:", list(xgb_scorecard.columns))
xgb_scorecard[["Tree", "Feature", "Sign", "Split", "XAddEvidence", "IV", "TreeSource"]]

XGBoost scorecard columns: ['Tree', 'Node', 'Feature', 'Sign', 'Split', 'Count', 'CountPct', 'NonEvents', 'Events', 'EventRate', 'WOE', 'IV', 'XAddEvidence', 'DetailedSplit', 'TreeSource']


,Tree,Feature,Sign,Split,XAddEvidence,IV,TreeSource
0,0,income,<,32296.250000,0.381748,0.549239,base
1,0,income,>=,32296.250000,-0.277945,1.582489,base
2,1,income,<,24092.572300,0.343839,0.633282,base
3,1,income,>=,24092.572300,-0.166161,0.436613,base
4,2,credit_age,<,2.341394,-0.295999,1.178644,base
5,2,credit_age,>=,2.341394,0.170014,0.202889,base
6,3,income,<,32296.250000,0.144004,0.549239,base
7,3,income,>=,32296.250000,-0.253732,1.582489,base
8,4,credit_age,<,2.341394,-0.284740,1.178644,base
9,4,credit_age,>=,2.341394,0.107195,0.202889,base


## 5. TreeSource Column

The `TreeSource` column identifies which trees came from the base model vs. fine-tuning.


In [7]:
# Show tree source distribution
print("XGBoost TreeSource distribution:")
print(xgb_scorecard["TreeSource"].value_counts())
print()

lgb_scorecard = lgb_sc.construct_scorecard()
print("LightGBM TreeSource distribution:")
print(lgb_scorecard["TreeSource"].value_counts())

XGBoost TreeSource distribution:
TreeSource
base         20
finetuned    10
Name: count, dtype: int64

LightGBM TreeSource distribution:
TreeSource
base         20
finetuned    10
Name: count, dtype: int64


## 6. Summarize Score Sources

See how Information Value is split between base and fine-tuned trees.


In [8]:
# IV contribution by source
print("XGBoost IV by Source")
print(xgb_sc.summarize_score_sources())

print("\nLightGBM IV by Source")
print(lgb_sc.summarize_score_sources())

print("\nCatBoost IV by Source")
print(cb_sc.summarize_score_sources())

XGBoost IV by Source
      Feature    BaseIV  FinetunedIV    TotalIV
0  credit_age  5.526132     1.381533   6.907665
1  debt_ratio  0.498784     0.498784   0.997568
2      income  8.160451     6.395184  14.555635

LightGBM IV by Source
      Feature    BaseIV  FinetunedIV    TotalIV
0  credit_age  5.142833     1.468016   6.610850
1  debt_ratio  0.488630     0.488630   0.977260
2      income  9.147110     2.073578  11.220688

CatBoost IV by Source
      Feature    BaseIV  FinetunedIV   TotalIV
0  credit_age  0.383577     0.625873  1.009450
1  debt_ratio  0.116040     0.000000  0.116040
2      income  1.042433     0.641951  1.684384


## 7. Score Comparison: Before vs. After Fine-Tuning


In [9]:
# Create points and predict scores
xgb_sc.create_points(pdo=50, target_points=600, target_odds=19)
scores_finetuned = xgb_sc.predict_score(X_test)

# Compare with base model scores
xgb_base_sc = XGBScorecardConstructor(xgb_base, X_train, y_train)
xgb_base_sc.construct_scorecard()
xgb_base_sc.create_points(pdo=50, target_points=600, target_odds=19)
scores_base = xgb_base_sc.predict_score(X_test)

comparison = pd.DataFrame(
    {
        "Base Score": scores_base.values,
        "Fine-tuned Score": scores_finetuned.values,
        "Actual": y_test.values,
    }
)
print("Score distribution comparison:")
print(comparison.describe().round(1))

Score distribution comparison:


       Base Score  Fine-tuned Score  Actual
count       150.0             150.0   150.0
mean        457.5             482.0     0.1
std          80.9              88.9     0.2
min         290.0             300.0     0.0
25%         423.2             442.0     0.0
50%         455.0             482.0     0.0
75%         486.0             510.0     0.0
max         587.0             625.0     1.0


## 8. SHAP Scoring with Fine-Tuned Models


In [10]:
# SHAP-based scoring works with fine-tuned models
shap_scores = xgb_sc.predict_score(X_test, method="shap")
shap_decomposition = xgb_sc.predict_scores(X_test, method="shap")

print("SHAP score summary:")
print(shap_scores.describe().round(1))
print("\nSHAP feature decomposition (first 5 rows):")
shap_decomposition.head()

SHAP score summary:
count    150.0
mean     435.6
std       89.9
min      252.0
25%      395.0
50%      436.0
75%      464.0
max      580.0
Name: score, dtype: float64

SHAP feature decomposition (first 5 rows):


,income_score,debt_ratio_score,credit_age_score,score
0,214,124,98,436
1,214,152,214,580
2,214,124,98,436
3,94,152,98,344
4,214,152,98,464


## 9. Alternative: Using `n_base_trees` Directly

You can also pass `n_base_trees` directly to the constructor without using `from_finetune_result()`.


In [11]:
# Direct construction with n_base_trees
manual_sc = XGBScorecardConstructor(xgb_ft.model, X_train, y_train, n_base_trees=10)
manual_scorecard = manual_sc.construct_scorecard()
print("TreeSource from manual n_base_trees:")
print(manual_scorecard["TreeSource"].value_counts())

TreeSource from manual n_base_trees:
TreeSource
base         20
finetuned    10
Name: count, dtype: int64
